0. Imports

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

1. Web Scraping

1.1. Obtaining the data from the CBF website and converting it into two lists (one with the column names and the other with the rows):

In [2]:
headers = []
rows = []

for i, ano in enumerate(range(2021, 2026)):
    url = f'https://www.cbf.com.br/futebol-brasileiro/tabelas/campeonato-brasileiro/serie-a/{ano}'
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Erro ao buscar {ano}")
        continue

    soup = BeautifulSoup(response.text, 'html.parser')
    content = soup.find('div', class_='styles_tableContent__Qbq2j')
    thead = content.find('thead')
    tbody = content.find('tbody')

    if not headers:
        headers = []
        for th in thead.find_all("th"):
            span = th.find("span")
            value = span.text.strip() if span else th.text.strip()
            if value != 'Recentes' and value != 'Próximo':
                headers.append(value)
        
        headers.append('Ano')

    for tr in tbody.find_all("tr"):
        row_in = []
        for td in tr.find_all("td"):
            strong_list = td.find_all("strong")
            value = [strong.text.strip() for strong in strong_list] if strong_list else td.text.strip()
            if value and strong_list:
                row_in.append(int(value[0]))
                row_in.append(value[1])
            elif value:
                row_in.append(int(value))

        row_in.append(ano)
        rows.append(row_in)

1.2. Adicionando times à lista headers:

In [3]:
headers.insert(1, "Time")
headers

['Classificação',
 'Time',
 'Pontos',
 'Jogos',
 'Vitórias',
 'Empates',
 'Derrotas',
 'Gols Prós',
 'Gols Contras',
 'Saldos de Gols',
 'Cartões Amarelos',
 'Cartões Vermelhos',
 'Aproveitamento',
 'Ano']

1.3. Transforming lists into dataframes:

In [4]:
seriesa_data = pd.DataFrame(rows, columns=headers)
seriesa_data.head()

,Classificação,Time,Pontos,Jogos,Vitórias,Empates,Derrotas,Gols Prós,Gols Contras,Saldos de Gols,Cartões Amarelos,Cartões Vermelhos,Aproveitamento,Ano
0,1,Atlético Mineiro,84,38,26,6,6,67,34,33,82,4,73,2021
1,2,Flamengo,71,38,21,8,9,69,36,33,82,4,62,2021
2,3,Palmeiras,66,38,20,6,12,58,43,15,87,11,57,2021
3,4,Fortaleza,58,38,17,7,14,44,45,-1,95,7,50,2021
4,5,Corinthians,57,38,15,12,11,40,36,4,61,3,50,2021


1.4. Determining the percentage of zero values:

In [5]:
(seriesa_data.isnull().sum() / len(seriesa_data)) * 100

Classificação        0.0
Time                 0.0
Pontos               0.0
Jogos                0.0
Vitórias             0.0
Empates              0.0
Derrotas             0.0
Gols Prós            0.0
Gols Contras         0.0
Saldos de Gols       0.0
Cartões Amarelos     0.0
Cartões Vermelhos    0.0
Aproveitamento       0.0
Ano                  0.0
dtype: float64

1.5. Standardizing different names that refer to the same team:

In [5]:
names = sorted(seriesa_data['Time'].unique())
names

['America',
 'America Saf',
 'Athletico Paranaense',
 'Atlético',
 'Atlético Goianiense Saf',
 'Atlético Mineiro',
 'Atlético Mineiro Saf',
 'Avaí',
 'Bahia',
 'Botafogo',
 'Ceará',
 'Chapecoense',
 'Corinthians',
 'Coritiba',
 'Coritiba S.a.f.',
 'Criciúma',
 'Cruzeiro Saf',
 'Cuiabá',
 'Cuiabá Saf',
 'Flamengo',
 'Fluminense',
 'Fortaleza',
 'Fortaleza Ec Saf',
 'Goiás',
 'Grêmio',
 'Internacional',
 'Juventude',
 'Mirassol',
 'Palmeiras',
 'Red Bull Bragantino',
 'Santos',
 'Santos Fc',
 'Sport',
 'Sport Recife',
 'São Paulo',
 'Vasco da Gama S.a.f.',
 'Vasco da Gama Saf',
 'Vitória']

In [ ]:
seriesa_data.loc[seriesa_data['Time'] == 'America', 'Time'] = 'America Saf'
seriesa_data.loc[seriesa_data['Time'] == 'Atlético', 'Time'] = 'Atlético Goianiense Saf'
seriesa_data.loc[seriesa_data['Time'] == 'Atlético Mineiro', 'Time'] = 'Atlético Mineiro Saf'
seriesa_data.loc[seriesa_data['Time'] == 'Coritiba', 'Time'] = 'Coritiba S.a.f.'
seriesa_data.loc[seriesa_data['Time'] == 'Cuiabá', 'Time'] = 'Cuiabá Saf'
seriesa_data.loc[seriesa_data['Time'] == 'Fortaleza', 'Time'] = 'Fortaleza Ec Saf'
seriesa_data.loc[seriesa_data['Time'] == 'Santos', 'Time'] = 'Santos Fc'
seriesa_data.loc[seriesa_data['Time'] == 'Sport', 'Time'] = 'Sport Recife'
seriesa_data.loc[seriesa_data['Time'] == 'Vasco da Gama S.a.f.', 'Time'] = 'Vasco da Gama Saf'

names = sorted(seriesa_data['Time'].unique())
names

['America Saf',
 'Athletico Paranaense',
 'Atlético Goianiense Saf',
 'Atlético Mineiro Saf',
 'Avaí',
 'Bahia',
 'Botafogo',
 'Ceará',
 'Chapecoense',
 'Corinthians',
 'Coritiba S.a.f.',
 'Criciúma',
 'Cruzeiro Saf',
 'Cuiabá Saf',
 'Flamengo',
 'Fluminense',
 'Fortaleza Ec Saf',
 'Goiás',
 'Grêmio',
 'Internacional',
 'Juventude',
 'Mirassol',
 'Palmeiras',
 'Red Bull Bragantino',
 'Santos Fc',
 'Sport Recife',
 'São Paulo',
 'Vasco da Gama Saf',
 'Vitória']

1.6. Downloading the dataframe in .csv format:

In [30]:
seriesa_data.to_csv("../../data/raw/brazilian_championship_seriesa.csv", index=False)